## 22127260 - Bùi Công Mậu
## 22127400 - Thái Hữu Thọ

In [ ]:
import pandas as pd
import tensorflow as tf
from sklearn.metrics import root_mean_squared_log_error
from sklearn.model_selection import train_test_split
import numpy as np
import time
import psutil
import os

In [ ]:
# Đọc file train CSV
df = pd.read_csv('train_normalized.csv')

# Log-transform target
df['target_log'] = np.log1p(df['target'])

# Tách features và target
X = df.drop(columns=['id', 'target', 'target_log'])
y = df['target_log']
y_original = df['target']

# Chia train/test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
y_test_original = np.expm1(y_test)

# Tạo tf.data.Dataset
train_dataset = tf.data.Dataset.from_tensor_slices((X_train.values, y_train.values))
train_dataset = train_dataset.cache().shuffle(buffer_size=len(X_train)).batch(32).prefetch(tf.data.AUTOTUNE)

test_dataset = tf.data.Dataset.from_tensor_slices((X_test.values, y_test.values))
test_dataset = test_dataset.batch(32).prefetch(tf.data.AUTOTUNE)

# Theo dõi thời gian bắt đầu và mức sử dụng CPU/RAM
start_time = time.time()
process = psutil.Process(os.getpid())

# Xây dựng mô hình với cải tiến
model = tf.keras.Sequential([
    tf.keras.layers.Dense(128, activation='relu', input_shape=(X_train.shape[1],)),  # Tăng số lượng neuron
    tf.keras.layers.Dropout(0.3),  # Thêm Dropout để giảm overfitting
    tf.keras.layers.BatchNormalization(),  # Thêm BatchNormalization
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dropout(0.3),  # Thêm Dropout nữa
    tf.keras.layers.Dense(1)  # Output là log(target)
])

# Compile với Nadam optimizer và điều chỉnh learning rate
model.compile(
    optimizer=tf.keras.optimizers.Nadam(learning_rate=0.0005),  # Sử dụng Nadam optimizer và giảm learning rate
    loss='mse',
    metrics=['mae']
)

# Sử dụng EarlyStopping để tránh overfitting
early_stopping = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

# Train mô hình
model.fit(
    train_dataset,
    epochs=100,
    validation_data=test_dataset,
    callbacks=[early_stopping],
    verbose=1
)

# Tính thời gian huấn luyện và bộ nhớ
end_time = time.time()
training_time = end_time - start_time
memory_usage = process.memory_info().rss / (1024 ** 2)  # in MB

print(f"Thời gian huấn luyện: {training_time:.2f} giây")
print(f"Mức sử dụng bộ nhớ: {memory_usage:.2f} MB")

# Dự đoán trên tập test (train) để kiểm tra RMSLE
y_pred_log = model.predict(test_dataset)
y_pred_original = np.expm1(y_pred_log.flatten())

# Tính RMSLE
rmsle = root_mean_squared_log_error(y_test_original, y_pred_original)
print(f"RMSLE trên tập test (mô hình cải tiến): {rmsle:.4f}")

# Đọc file test_normalized.csv (80,000 dòng)
test_df = pd.read_csv('test_normalized.csv')

# Tách features từ test_df (bỏ cột id)
X_test_new = test_df.drop(columns=['id'])

# Dự đoán trên tập test mới
test_dataset_new = tf.data.Dataset.from_tensor_slices(X_test_new.values)
test_dataset_new = test_dataset_new.batch(32).prefetch(tf.data.AUTOTUNE)

y_pred_log_new = model.predict(test_dataset_new)
y_pred_new = np.expm1(y_pred_log_new.flatten())  # Chuyển từ log-target về target gốc

# Tạo DataFrame kết quả với 2 cột: id và target
result_df = pd.DataFrame({
    'id': test_df['id'],
    'target': y_pred_new
})

# Lưu vào file CSV
result_df.to_csv('22127260_22127400.csv', index=False)
print("Đã lưu kết quả dự đoán vào file '22127260_22127400.csv'")


Epoch 1/100


c:\Users\LAPTOP HP\anaconda3\Lib\site-packages\keras\src\layers\core\dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


30000/30000 ━━━━━━━━━━━━━━━━━━━━ 63s 2ms/step - loss: 2.5485 - mae: 1.1587 - val_loss: 1.1874 - val_mae: 0.7983
Epoch 2/100
30000/30000 ━━━━━━━━━━━━━━━━━━━━ 54s 2ms/step - loss: 1.2808 - mae: 0.8414 - val_loss: 1.1826 - val_mae: 0.7887
Epoch 3/100
30000/30000 ━━━━━━━━━━━━━━━━━━━━ 55s 2ms/step - loss: 1.2132 - mae: 0.8071 - val_loss: 1.1797 - val_mae: 0.7918
Epoch 4/100
30000/30000 ━━━━━━━━━━━━━━━━━━━━ 54s 2ms/step - loss: 1.1904 - mae: 0.7946 - val_loss: 1.1751 - val_mae: 0.7844
Epoch 5/100
30000/30000 ━━━━━━━━━━━━━━━━━━━━ 56s 2ms/step - loss: 1.1783 - mae: 0.7882 - val_loss: 1.1779 - val_mae: 0.7952
Epoch 6/100
30000/30000 ━━━━━━━━━━━━━━━━━━━━ 52s 2ms/step - loss: 1.1815 - mae: 0.7880 - val_loss: 1.1720 - val_mae: 0.7807
Epoch 7/100
30000/30000 ━━━━━━━━━━━━━━━━━━━━ 56s 2ms/step - loss: 1.1762 - mae: 0.7853 - val_loss: 1.1721 - val_mae: 0.7820
Epoch 8/100
30000/30000 ━━━━━━━━━━━━━━━━━━━━ 55s 2ms/step - loss: 1.1764 - mae: 0.7855 - val_loss: 1.1707 - val_mae: 0.7772
Epoch 9/100
30000/30